Foundry Skills live in git. That's great for developers, but most of the
people who actually know whether a skill is *right* ? a support lead who
knows the real refund policy, a product manager who knows what a loyalty
program should say ? don't use git, don't have a GitHub habit, and
shouldn't have to learn one just to approve a two-sentence wording change.

This recipe walks through a sample we built to close that gap: a Foundry
hosted agent, published to Microsoft Teams, that lets business users
review skill pull requests, comment, approve, merge, and propose their
own changes ? all from a normal chat, with every action attributed to
*their own* GitHub account.


## 1 / Why we built this

Skills are code review artifacts whether we like it or not. If you store
Foundry Skills as files in a repo (which is the natural place for them ?
versioned, diffable, revertable), then changing a skill means opening a
pull request, and approving a skill means approving that PR. That's fine
for engineers. It's a wall for everyone else.

The obvious first fix ? "let an agent do it for them" ? has an obvious
trap. If the agent uses one shared GitHub token to list PRs, comment,
approve, and merge on behalf of *every* Teams user, you've built a system
where the audit trail says "the bot did it," not "Alice approved it" or
"Bob proposed it." That's not just unsatisfying, it's actually unsafe:
GitHub's own protections ? "require an approving review," "you can't
approve your own PR" ? are keyed off the identity making the API call. A
shared token can't be checked by two different people, and it can approve
its own proposals all day long. We tried building it that way first. It
worked, technically, and then we realized it worked around the one
guarantee that actually matters.

So the real requirement wasn't "let business users talk to GitHub through
an agent." It was: **every git action a Teams user takes through this bot
must be authenticated as that person, on GitHub, with no exceptions** ?
and the bot itself should never need to know or store anyone's GitHub
credentials.


## 2 / How we built it

### The shape of it

```text
Business user in Teams
  -> Foundry hosted agent (Agent Framework, Python)
  -> Foundry toolbox (MCP) -> GitHub's catalog MCP server
       https://api.githubcopilot.com/mcp, OAuth2, per Teams user
```

The agent itself is a small Agent Framework `Agent`. It has exactly one
tool: a `FoundryToolbox` pointed at a Foundry **project connection** that
targets GitHub's own MCP server. No custom REST wrappers, no
`GITHUB_TOKEN`, no bot identity at all.


In [ ]:
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient, FoundryToolbox, ResponsesHostServer

# Resolved from FOUNDRY_PROJECT_ENDPOINT + TOOLBOX_NAME env vars.
toolbox = FoundryToolbox(credential)

agent = Agent(
    client=client,
    instructions=INSTRUCTIONS,  # see section 2 below - plain-language, no git jargon
    tools=toolbox,
)


### Per-user auth, for free

Here's the part that made this worth writing up. `FoundryToolbox` forwards
a platform call-id header on every request. The Foundry MCP proxy uses
that header to resolve *which Teams user* is making the call, and swaps in
that user's own GitHub OAuth token before hitting
`api.githubcopilot.com/mcp`. We didn't write any of that plumbing. The
agent code has zero awareness of who's asking ? it just calls
`gh___pull_request_read`, `gh___merge_pull_request`,
`gh___create_pull_request`, and so on, and the identity attached to each
call is whoever is actually chatting.

The connection itself is a standard catalog-MCP, managed-OAuth project
connection:


In [ ]:
connection_body = {
    "properties": {
        "authType": "OAuth2",
        "category": "RemoteTool",
        "target": "https://api.githubcopilot.com/mcp",
        "connectorName": "foundrygithubmcp",
        "metadata": {
            "type": "catalog_MCP",
            "toolEntityId": (
                "azureml://location/eastus/apiCenter/connectors-registry-prod-bl"
                "/type/tools/objectId/github/version/1"
            ),
        },
    }
}
# PUT this to
# {ARM}/.../projects/{proj}/connections/{name}?api-version=2025-04-01-preview


wired into a toolbox that the agent references by name. Consent isn't
something you call manually ? the first time a given Teams user's message
needs GitHub, the toolbox call comes back with a `CONSENT_REQUIRED` error
carrying a one-time GitHub sign-in link. The agent just shows that link to
the user. Once they sign in, every following call in that conversation ?
and every future conversation ? runs as them. We verified this the
boring, reliable way: by calling `gh___get_me` through the toolbox and
getting back the signer's own GitHub login, not a shared service account.

### The merge gate had to move

The nice thing about writing your own REST wrappers is you can put a gate
in front of anything ? e.g. "check for an approving review before I'll
even call the merge endpoint." The moment GitHub access becomes a generic
MCP toolbox instead of your own Python functions, that code-level
interception disappears; the model is calling GitHub's tools directly.

We replaced it with two layers instead of one:

1. **Instructions**, telling the model to read a PR's reviews
   (`gh___pull_request_read`) before ever calling
   `gh___merge_pull_request`, and to refuse if there's no approval or an
   outstanding "request changes."
2. **GitHub branch protection**, which is the layer that actually can't be
   talked around. "Require a pull request review before merging" makes
   GitHub itself reject the call regardless of what the model decides.
   GitHub also refuses self-approval outright ("you can't approve your
   own pull request") ? another guarantee that only works because the
   call is attributed to a real person.

Software-only enforcement (1) is a nice UX layer. Enforcement (2) is the
one you should actually rely on. That's worth saying plainly, because it's
tempting to assume "the agent's instructions say so" is a security
control. It isn't ? treat it as a courtesy on top of GitHub's real rules.

### Talking to business users like they're business users

The other lesson, once we started testing this with a non-developer
mindset: don't make them speak git. Early drafts of the instructions still
asked things like "what should I name the branch?" or "what's the PR
title?" ? exactly the kind of question that stops a support lead cold.
The fix was to push all of that down into the agent's own judgment:

- Ask only about *intent* ? which skill, what should change ? never about
  branch names, commit messages, or PR titles. The agent generates all of
  those itself from the skill name and a short summary.
- If someone asks what "merge" or "pull request" or "branch" means,
  explain it in one plain sentence, and only when asked or clearly needed
  ? not as an unprompted lecture.
- Don't guess at things the underlying GitHub tools don't actually support
  (we initially told the model to pass a `labels` argument straight into
  `create_pull_request`; that tool has no such parameter, only
  `reviewers` ? the actual GitHub MCP tool schema is the source of truth,
  not what seems like it should exist). Apply labels afterwards with
  `gh___issue_write` instead, since a pull request is addressable as an
  issue for labeling purposes.

None of this needed new code. It's all in the system instructions. Which
is the point: once the toolbox handles auth and GitHub handles
enforcement, the agent's job is almost entirely "translate between how a
human describes a problem and which two or three tool calls solve it."


## 3 / How to get started

The full sample ? code, README, and setup steps ? is
[`18-skill-review-teams-bot`](https://github.com/microsoft-foundry/foundry-samples/tree/main/samples/python/hosted-agents/agent-framework/responses/18-skill-review-teams-bot)
in the Foundry samples repo. Here's the short version:

1. **Create the GitHub project connection and toolbox once**, per project
   (see the sample README for the exact PUT/POST bodies):
   - An `OAuth2` / `catalog_MCP` project connection targeting
     `https://api.githubcopilot.com/mcp`.
   - A toolbox that attaches that connection, promoted to `default`.
2. **Write the agent.** Construct a `FoundryToolbox(credential)`, pass it
   as the agent's `tools`, and write instructions in terms of what your
   users are trying to do, not which git operations exist. Keep a
   glossary of one-line, plain-language explanations for
   branch/commit/PR/diff/merge/fork ? you'll want it the first time
   someone asks.
3. **Deploy with `azd deploy`.** Standard Agent Framework hosted-agent
   flow ? no extra steps for the toolbox, since it's a project-level
   resource the agent just references by name.
4. **Publish to Teams** from the Foundry portal (**Publish -> Publish to
   Teams and Microsoft 365**). Business users sign in once for the bot
   itself, and once for GitHub the first time they trigger a GitHub
   action ? after that, it's just chat.
5. **Set up branch protection** on your skills repo: require an approving
   review before merge. This is what actually enforces "two people must
   agree," not anything the agent says.

If you're building anything similar ? any Teams-facing agent that needs
to act on a developer-facing system on behalf of a business user ? the
pattern generalizes past GitHub: put a per-user OAuth2/catalog-MCP
connection in front of it, let the platform resolve identity per caller,
and keep your agent's instructions focused on translating intent rather
than asking users to speak the underlying system's language.
